# CICIoT2023 — XGBoost IoT Saldırı Tespit Modeli Eğitimi

**Proje:** Web Arayüzlü ve Gerçeğe Yakın Zamanlı Anomali Tespit Sistemi (HIDS-XGBoost-SHAP-SLM)

**Hedef:** Raspberry Pi 4 (8GB) üzerinde gerçek zamanlıya yakın çalışacak, hafif ve hızlı bir XGBoost saldırı tespit modeli eğitmek.

**Veri Seti:** CICIoT2023 (Canadian Institute for Cybersecurity, 2023)
- 105 gerçek IoT cihazı (akıllı kamera, hoparlör, sensör, akıllı priz)
- 33 modern saldırı türü, 7 kategori: DDoS, DoS, Recon, Web-based, Brute Force, Spoofing, Mirai
- Akış-bazlı öznitelikler (paket sayısı, byte oranı, inter-arrival time)

**Neden CICIoT2023?**
- Akıllı ev senaryosuyla birebir örtüşür (projemizin konusu)
- 2023 verisi — NSL-KDD (1999) gibi eski değil, modern saldırılar içerir
- Literatürde XGBoost ile %99+ başarı kanıtlanmış [Neto et al., 2023]

**Çıktılar:**
- `xgboost_ciciot2023.joblib` — Pi'ye taşınacak eğitimli model
- `scaler.joblib` — Öznitelik normalizasyonu
- `feature_names.json` — Öznitelik sırası (canlı tespit için kritik)
- SHAP açıklanabilirlik grafikleri


## 0. Kurulum ve Kütüphaneler

In [ ]:
!pip install -q xgboost scikit-learn shap imbalanced-learn kagglehub joblib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json, os, time, glob, gc

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, accuracy_score, roc_auc_score)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

print("XGBoost:", xgb.__version__)
print("SHAP:", shap.__version__)


## 1. CICIoT2023 Veri Setini İndir

Kaggle üzerinden indiriyoruz. Veri seti büyük (~13GB ham); Kaggle'ın hazır CSV
sürümünü kullanıyoruz. İlk çalıştırmada Kaggle API anahtarı istenebilir.

**Alternatif:** Veri seti çok büyükse, CSV'lerin bir alt kümesiyle (örn. ilk 10
dosya) çalışmak Pi-hedefli hafif model için yeterlidir.


In [ ]:
import kagglehub

# CICIoT2023 (flow-based CSV sürümü)
path = kagglehub.dataset_download("madhavmalhotra/unb-cic-iot-dataset")
print("Veri seti konumu:", path)

# CSV dosyalarını bul
csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
print(f"Bulunan CSV dosyası: {len(csv_files)}")
for f in csv_files[:5]:
    print(" ", os.path.basename(f))


## 2. Veri Yükleme ve Örnekleme

CICIoT2023 çok büyük (46M+ kayıt). Pi-hedefli hafif model için dengeli bir
**alt küme** kullanıyoruz: her dosyadan örnek alıp birleştiriyoruz. Bu hem
eğitimi hızlandırır hem de bellek taşmasını önler.


In [ ]:
# Bellek dostu yükleme: her dosyadan örnek al
MAX_FILES = 20          # İşlenecek CSV sayısı (artırılabilir)
ROWS_PER_FILE = 50000   # Her dosyadan alınacak satır

dfs = []
for i, f in enumerate(csv_files[:MAX_FILES]):
    try:
        df = pd.read_csv(f, nrows=ROWS_PER_FILE)
        dfs.append(df)
        print(f"[{i+1}/{MAX_FILES}] {os.path.basename(f)} -> {df.shape}")
    except Exception as e:
        print(f"Atlandı {os.path.basename(f)}: {e}")

data = pd.concat(dfs, ignore_index=True)
del dfs; gc.collect()
print("\nToplam veri:", data.shape)
data.head()


## 3. Ön İşleme ve Öznitelik Seçimi (Canlı Uyum)

- Etiket sütununu bul; **Binary dönüşüm:** BENIGN/NORMAL = 0, diğer her şey = 1 (Saldırı)
- **AKADEMİK DÜRÜSTLÜK / train-serve parity:** Model YALNIZCA canlı sniffer'ın
  (`hids-sensor/sensor/flow_aggregator.py` → `FEATURE_NAMES`) sadık biçimde
  üretebildiği **40 öznitelikle** eğitilir. Veri setinin çıkarıcısıyla tanımını
  birebir eşleyemediğimiz kompozitler (`Magnitue`, `Radius`, `Covariance`,
  `Weight`, `Duration` (TTL), `Tot size`) **kasıtlı olarak dışarıda bırakılır** —
  canlıda uydurma/yaklaşık değer üretmemek için. Böylece eğitim ile canlı çıkarım
  arasında öznitelik kayması (train/serve skew) ortadan kalkar.
- Sonsuz/eksik değerleri temizle; RobustScaler (aykırı değerlere dayanıklı).

> Binary sınıflandırma seçildi: canlı HIDS'te asıl soru "saldırı var mı?". Çok-sınıflı
> tespit (hangi saldırı türü) ileri çalışma olarak bırakıldı.


In [ ]:
# Etiket sütununu otomatik bul
label_col = None
for cand in ['label', 'Label', 'LABEL', 'class', 'Class']:
    if cand in data.columns:
        label_col = cand
        break
if label_col is None:
    label_col = data.columns[-1]  # Son sütun varsayılan
print("Etiket sütunu:", label_col)
print("\nSınıf dağılımı:")
print(data[label_col].value_counts().head(10))


In [ ]:
# ── Öznitelik seçimi: SADECE canlı ortamda sadık hesaplanabilen öznitelikler ──
# Bu liste flow_aggregator.py içindeki FEATURE_NAMES ile BİREBİR aynı (isim + sıra)
# olmalıdır. Aksi halde canlı çıkarım ile eğitim arasında uyumsuzluk doğar.
LIVE_FEATURES = [
    "flow_duration", "Header_Length", "Protocol Type",
    "Rate", "Srate", "Drate",
    "fin_flag_number", "syn_flag_number", "rst_flag_number",
    "psh_flag_number", "ack_flag_number", "ece_flag_number", "cwr_flag_number",
    "ack_count", "syn_count", "fin_count", "urg_count", "rst_count",
    "HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC",
    "TCP", "UDP", "DHCP", "ARP", "ICMP", "IPv", "LLC",
    "Tot sum", "Min", "Max", "AVG", "Std", "IAT", "Number", "Variance",
]

# Binary etiket: BENIGN/Normal -> 0, saldırı -> 1
def to_binary(label):
    s = str(label).upper()
    return 0 if ('BENIGN' in s or 'NORMAL' in s) else 1

y = data[label_col].apply(to_binary).values

# Veri setinde bulunan LIVE_FEATURES sütunlarını AYNI SIRADA seç
present = [c for c in LIVE_FEATURES if c in data.columns]
missing = [c for c in LIVE_FEATURES if c not in data.columns]
if missing:
    print("UYARI: Veri setinde bulunamayan öznitelikler:", missing)
    print(">>> Sütun adlarını kontrol edin; eksik sütun canlı tarafla pariteyi bozar.")
else:
    print(f"Tüm {len(LIVE_FEATURES)} canlı-uyumlu öznitelik veri setinde mevcut.")

X = data[present].copy()
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

feature_names = present  # canlı tarafla (flow_aggregator.FEATURE_NAMES) BİREBİR aynı
print(f"Kullanılan öznitelik sayısı: {len(feature_names)}")
print(f"Normal: {(y==0).sum():,} | Saldırı: {(y==1).sum():,}")


## 4. Train/Test Bölme, Ölçekleme ve SMOTE

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y, test_size=0.2, random_state=42, stratify=y)

# Ölçekleme
scaler = RobustScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# SMOTE — yalnızca eğitim setine, sınıf dengesizliği için
if (y_train == 0).sum() > 5 and (y_train == 1).sum() > 5:
    sm = SMOTE(random_state=42)
    X_train, y_train = sm.fit_resample(X_train, y_train)
    print("SMOTE sonrası eğitim:", X_train.shape)

print("Eğitim:", X_train.shape, "| Test:", X_test.shape)


## 5. XGBoost Eğitimi (Raspberry Pi Hedefli)

**Pi optimizasyonu:** Modeli hafif tutmak için `max_depth` ve `n_estimators`
sınırlı. Daha az ağaç + daha sığ derinlik = daha küçük model + daha hızlı
çıkarım. Pi 4'te TreeSHAP ve tahmin milisaniyeler içinde çalışır.


In [ ]:
model = xgb.XGBClassifier(
    n_estimators=150,        # Pi için makul (200 yerine)
    max_depth=8,             # Sığ ağaçlar -> küçük model
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    device='cuda',
    tree_method='hist',      # Hızlı eğitim
    eval_metric='logloss',
    random_state=42,
)

t0 = time.time()
model.fit(X_train, y_train)
print(f"GPU Destekli Eğitim Süresi: {time.time()-t0:.1f} saniye")


## 6. Değerlendirme

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

# FPR
tn = ((y_pred == 0) & (y_test == 0)).sum()
fp = ((y_pred == 1) & (y_test == 0)).sum()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

print(f"Accuracy : {acc*100:.2f}%")
print(f"F1-Score : {f1*100:.2f}%")
print(f"AUC-ROC  : {auc*100:.2f}%")
print(f"FPR      : {fpr*100:.2f}%")
print("\n", classification_report(y_test, y_pred, target_names=['Normal','Saldiri']))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal','Saldiri'], yticklabels=['Normal','Saldiri'])
plt.title('CICIoT2023 — XGBoost Confusion Matrix')
plt.ylabel('Gerçek'); plt.xlabel('Tahmin')
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=150); plt.show()


## 7. SHAP Açıklanabilirlik Analizi

SHAP (SHapley Additive exPlanations), modelin hangi özniteliklere dayanarak
karar verdiğini gösterir. Bu, projenin **açıklanabilir yapay zeka (XAI)**
bileşenidir ve canlı sistemde "neden anomali?" sorusuna cevap verir.

TreeSHAP, XGBoost ile çok hızlıdır ve Pi'de çalışabilir.


In [ ]:
# Hız için test setinden örnek al
sample = X_test[:1000]
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(sample)

# Beeswarm — öznitelik etkisi
shap.summary_plot(shap_values, sample, feature_names=feature_names,
                  max_display=15, show=False)
plt.tight_layout(); plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# Bar — öznitelik önem sıralaması
shap.summary_plot(shap_values, sample, feature_names=feature_names,
                  plot_type='bar', max_display=15, show=False)
plt.tight_layout(); plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight'); plt.show()

# En önemli 15 özniteliği kaydet (canlı sistemde bunlara odaklanılabilir)
importance = np.abs(shap_values).mean(axis=0)
top_features = sorted(zip(feature_names, importance), key=lambda x: -x[1])[:15]
print("En önemli 15 öznitelik:")
for name, imp in top_features:
    print(f"  {name}: {imp:.4f}")


## 8. Modeli Kaydet ve İndir

Bu dosyaları Raspberry Pi'ye (veya host PC'ye) taşıyıp `detector.py` içinde
kullanacağız. **`feature_names.json` kritik** — canlı sniffer'dan gelen
öznitelikler bu sırayla modele verilmelidir.


In [ ]:
# Model boyutunu küçük tutmak için joblib compress
joblib.dump(model, 'xgboost_ciciot2023.joblib', compress=3)
joblib.dump(scaler, 'scaler.joblib', compress=3)

with open('feature_names.json', 'w') as f:
    json.dump(feature_names, f, indent=2)

# Model metaverisi
meta = {
    "dataset": "CICIoT2023",
    "model": "XGBoost",
    "n_features": len(feature_names),
    "n_estimators": 150,
    "max_depth": 8,
    "accuracy": float(acc),
    "f1_score": float(f1),
    "auc_roc": float(auc),
    "fpr": float(fpr),
    "task": "binary (normal/attack)",
    "feature_policy": "live-computable subset (no fabricated composites)",
}
with open('model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

import os
size_mb = os.path.getsize('xgboost_ciciot2023.joblib') / 1e6
print(f"Model boyutu: {size_mb:.2f} MB  (Pi 4 için uygun)")
print("Kaydedilen dosyalar:")
for fn in ['xgboost_ciciot2023.joblib','scaler.joblib','feature_names.json','model_meta.json']:
    print("  ", fn)


In [ ]:
# Çıkarım hızı testi (Pi performansı tahmini)
import time
single = X_test[:1]
t0 = time.time()
for _ in range(1000):
    model.predict(single)
ms = (time.time()-t0)
print(f"1000 tahmin: {ms*1000:.1f} ms -> tahmin başına {ms:.3f} ms")
print("Not: Pi 4 CPU'da bu ~5-10x daha yavaş olur ama yine de <5ms/tahmin beklenir.")


In [ ]:
# Colab'dan indir
from google.colab import files
files.download('xgboost_ciciot2023.joblib')
files.download('scaler.joblib')
files.download('feature_names.json')
files.download('model_meta.json')


## Sonraki Adımlar

1. İndirilen 4 dosyayı `HIDS-XGBoost-SHAP-SLM/models/` klasörüne koy
2. `detector.py`'yi `feature_names.json` sırasına göre öznitelik çıkaracak şekilde güncelle
3. Canlı sniffer çıktısını modele besle, gerçek trafikte test et
4. SHAP'ı canlı sisteme entegre et (her anomali için açıklama)

**Önemli not:** CICIoT2023 akış-bazlı öznitelikler kullanır (flow features).
Canlı sniffer'ımız şu an paket-bazlı çalışıyor. İki seçenek:
- (A) Sniffer'a akış toplama (flow aggregation) ekle — daha doğru
- (B) Paket-bazlı basit özniteliklerle ayrı bir model eğit — daha hızlı
Bu kararı bir sonraki aşamada vereceğiz.
